# Data processing

As **input**, this script takes raw timeseries from the stations and models, saved in .nc files:
- Separately for each model
- Exactly 10 years for models, up to 30 years for observations
- Precipitation and temperature
- Hourly resolution
- Only station locations, i.e. for the models it's not gridded data, but timeseries from 277 grid cells that are nearest neighbors of the stations
- The files were processed to exclude obivous outliers, e.g. timesteps with more than 100mm/h averaged over all stations (i.e., large scale). Similar outliers were found and removed from station data (precipitation over ~200 mm/h). It is possible that some outliers remain.

As **output**, this script produces two types of auxiliary .nc files where intermediate results are stored:
- **extended hourly timeseries**: dataset with hourly frequency which contains:
    - hourly precipitation timeseries
    - hourly temperature timeseries
    - mean daily temperature timeseries broadcasted to an hourly frequency
    - total daily precipitation timeseries broadcasted to hourly frequency
- **extended daily timeseries**: dataset which contains the following timeseries with daily frequency:
    - 'mean_daily_tas' (average temperature over 24 hours)
    - 'pr' (total daily precipitation sum over 24 hours)
    - 'wet_hour_count' (number of wet hours within the day, used to determine wet hour freqnency later in the analysis)
    - 'wet_hour_mean_intensity' (how much precipitation falls within an hour if the hour is wet?
        Note: this is calculated separately for each day here; nan if there are zero wet hours in a day)
    - 'wet_hour_max_intensity' (what's the most intense precipitation that falls in that day?
        Note: this in NOT equivalent to high percentiles as this is calculated separately for each day;
        nan if there are zero wet hours in a day)
    - 'pr_onset_time' (time of precipitation onset, i.e. time in UTC when the first wet hour occurs.
        This is calculated only for days with at least one wet hour)

The following .nc files are produced from the extended timeseries and serve as the main output of this script, used in later processing:
- **quantiles_DatasetName_TimePeriod_hdmean_Season.nc**: "hdmean" stands for "hourly precipitation, daily mean tempearture". 
    The files contain hourly precipitation quantiles, count of wet hours, and count of all hours in each temperature bin.
- **quantiles_DatasetName_TimePeriod_ddmean_Season.nc**: "ddmean" stands for "daily precipitation, daily mean tempearture". 
    The files contain daily precipitation quantiles, count of wet days, and count of all days in each temperature bin.
- **various_daily_stats_DatasetName_TimePeriod_dmean_Season.nc**: "dmean" stands for "daily mean tempearture". 
    The files contain mean daily precipitation, count of days at each temperature, mean and maximum of wet hour intensities,
        time of onset of precipitation, as well as all these aforementioned characteristics separated by 
        various ranges of mean daily precipitation (0.1-1, 1-2.5, 25.-5, 5-10, 10+ mm/d).
        
One file is produced per time period (evaluation/historical/rcp), per season (DJF/MAM/JJA/SON/all year), per model.
Subsequently, ensemble files are created, containing all the models per time period and per season.

**Note**: currently, this script creates 185 GB of additional processed data.

This could be reduced to ~100 GB with relatively simple code changes: instead of saving "extended_daily_measures_\*" and "extended_hourly_measures_\*" separately for all seasons (allseasons) and each individual season (DJF, MAM, JJA, SON), this could be done for all seasons once, and season selection would follow afterwards (function make_all_datasets_from_hourly_data())

In [1]:
import concurrent.futures
import glob
import os

import xarray as xr

In [2]:
from utils import make_all_datasets_from_hourly_data_parallel

# Definitions

In [3]:
# to run the script, change this based on where you saved the data
input_folder = "."
input_folder = "/gpfs/data/fs71966/amedvedova/publication_data"

# paths to data: this code cell has to be adjusted to where your files are saved if you want to reuse the script.
# parent folder where all raw station and model data is saved
folder_raw_data = f"{input_folder}/raw_data"

# path to the folder where processed files will be saved
folder_processed_data = f"{input_folder}/processed_data"

# flag to save files: set to false for development/debugging reasons
savefiles = True

if savefiles:
    # if the output folder doesn't exist yet, create it
    os.makedirs(folder_processed_data, exist_ok=True)

# cores to use in parallelized processing
workers = 64

In [4]:
# station data
filepath_geosphere = f"{folder_raw_data}/station_timeseries_geosphere_v2_relh_sorted_1990-2020.nc"
ds_geosphere = xr.open_dataset(filepath_geosphere, engine='h5netcdf')

# metadata to be added to the processed files
metadata = ds_geosphere[["lat", "lon", "elevation", "station_id"]]

In [5]:
# all evaluation files: km-scale + driving (coarse)
files_kmscale_eval = sorted(
    glob.glob(f"{folder_raw_data}/ALP-3/evaluation/*/*1hr_complete.nc")
)
files_upscale_eval = sorted(
    glob.glob(f"{folder_raw_data}/ALP-12/evaluation/*/*1hr_complete.nc")
)
files_driving_eval = sorted(
    glob.glob(f"{folder_raw_data}/EUR-12/evaluation/*/*1hr_complete.nc")
)

# all historical and RCP8.5 files: km-scale + driving (coarse)
# include CLMcom-ETH-GAR/REU from eval in hist, and include pgw in rcp:
# the eval/PGW pairs of these two models imitate hist+rcp scenarios in other models
files_kmscale_hist = sorted(
    glob.glob(f"{folder_raw_data}/ALP-3/historical/*/*1hr_complete.nc") +
    glob.glob(f"{folder_raw_data}/ALP-3/evaluation/CLMcom-ETH-*/*1hr_complete.nc")
)
files_upscale_hist = sorted(
    glob.glob(f"{folder_raw_data}/ALP-12/historical/*/*1hr_complete.nc") +
    glob.glob(f"{folder_raw_data}/ALP-12/evaluation/CLMcom-ETH-*/*1hr_complete.nc")
)
files_driving_hist = sorted(
    glob.glob(f"{folder_raw_data}/EUR-12/historical/*/*1hr_complete.nc") +
    glob.glob(f"{folder_raw_data}/EUR-12/evaluation/CLMcom-ETH-*/*1hr_complete.nc")
)

files_kmscale_rcp = sorted(
    glob.glob(f"{folder_raw_data}/ALP-3/rcp85/*/*1hr_complete.nc") +
    glob.glob(f"{folder_raw_data}/ALP-3/PGW-MPI-rcp85/*/*1hr_complete.nc")
)
files_upscale_rcp = sorted(
    glob.glob(f"{folder_raw_data}/ALP-12/rcp85/*/*1hr_complete.nc") +
    glob.glob(f"{folder_raw_data}/ALP-12/PGW-MPI-rcp85/*/*1hr_complete.nc")
)
files_driving_rcp = sorted(
    glob.glob(f"{folder_raw_data}/EUR-12/rcp85/*/*1hr_complete.nc") +
    glob.glob(f"{folder_raw_data}/EUR-12/PGW-MPI-rcp85/*/*1hr_complete.nc")
)

In [6]:
# season names and the corresponding months
seasons = ["allseasons", "DJF", "MAM", "JJA", "SON", "allseasons"]
months = [None, [12, 1, 2], [3, 4, 5], [6, 7, 8], [9, 10, 11]]

# Make files

## Files per model

In [ ]:
# GeoSphere (station data)
# for some reason this doesn't work in parallel:the JupyterHub server crashes after every season and has to be restarted.
# for season_months, season in zip(months, seasons):
#     make_all_datasets_from_hourly_data_parallel(filepath_geosphere, season_months, season, metadata, folder_processed_data=folder_processed_data, savefiles=savefiles, period="eval")

# if needed, restart the server after every season and comment out the processed line
make_all_datasets_from_hourly_data_parallel(filepath_geosphere, months[0], seasons[0], metadata, folder_processed_data=folder_processed_data, savefiles=savefiles, period="eval")
make_all_datasets_from_hourly_data_parallel(filepath_geosphere, months[1], seasons[1], metadata, folder_processed_data=folder_processed_data, savefiles=savefiles, period="eval")
make_all_datasets_from_hourly_data_parallel(filepath_geosphere, months[2], seasons[2], metadata, folder_processed_data=folder_processed_data, savefiles=savefiles, period="eval")
make_all_datasets_from_hourly_data_parallel(filepath_geosphere, months[3], seasons[3], metadata, folder_processed_data=folder_processed_data, savefiles=savefiles, period="eval")
make_all_datasets_from_hourly_data_parallel(filepath_geosphere, months[4], seasons[4], metadata, folder_processed_data=folder_processed_data, savefiles=savefiles, period="eval")

make_all_datasets_from_hourly_data_parallel(filepath_geosphere, months[0], seasons[0], metadata, folder_processed_data=folder_processed_data, savefiles=savefiles, period="eval", groupby_var="mean_daily_td")
make_all_datasets_from_hourly_data_parallel(filepath_geosphere, months[1], seasons[1], metadata, folder_processed_data=folder_processed_data, savefiles=savefiles, period="eval", groupby_var="mean_daily_td")
make_all_datasets_from_hourly_data_parallel(filepath_geosphere, months[2], seasons[2], metadata, folder_processed_data=folder_processed_data, savefiles=savefiles, period="eval", groupby_var="mean_daily_td")
make_all_datasets_from_hourly_data_parallel(filepath_geosphere, months[3], seasons[3], metadata, folder_processed_data=folder_processed_data, savefiles=savefiles, period="eval", groupby_var="mean_daily_td")
make_all_datasets_from_hourly_data_parallel(filepath_geosphere, months[4], seasons[4], metadata, folder_processed_data=folder_processed_data, savefiles=savefiles, period="eval", groupby_var="mean_daily_td")

In [39]:
# Model data
# get a list of files and corresponding time periods to be passed further
# model_period_pairs = [
#     (files_kmscale_eval, "eval"),
#     (files_upscale_eval, "eval"),
#     (files_driving_eval, "eval"),
#     (files_kmscale_hist, "hist"),
#     (files_upscale_hist, "hist"),
#     (files_driving_hist, "hist"),
#     (files_kmscale_rcp, "rcp"),
#     (files_upscale_rcp, "rcp"),
#     (files_driving_rcp, "rcp"),        
#     ]
model_period_pairs = [
    (files_upscale_eval, "eval"),
    (files_upscale_hist, "hist"),
    (files_upscale_rcp, "rcp"),
    ]


# process data
if __name__ == '__main__':
    with concurrent.futures.ProcessPoolExecutor(max_workers=64) as exe:
        for season_months, season in zip(months, seasons):
            for files, period in model_period_pairs:
                for file_in in files:
                    result = exe.submit(make_all_datasets_from_hourly_data_parallel, file_in, season_months, season, metadata, folder_processed_data=folder_processed_data, savefiles=savefiles, period=period)

allseasons upscale_CLMcom-KIT rcp
/gpfs/data/fs71966/amedvedova/publication_data/raw_data/ALP-12/rcp85/CLMcom-KIT/ALP-12_rcp85_CLMcom-KIT_10yr_1hr_complete.nc
allseasons upscale_FZJ-IDL-CA rcp
/gpfs/data/fs71966/amedvedova/publication_data/raw_data/ALP-12/rcp85/FZJ-IDL-CA/ALP-12_rcp85_FZJ-IDL-CA_10yr_1hr_complete.nc
allseasons upscale_CLMcom-WEGC rcp
/gpfs/data/fs71966/amedvedova/publication_data/raw_data/ALP-12/rcp85/CLMcom-WEGC/ALP-12_rcp85_CLMcom-WEGC_10yr_1hr_complete.nc
allseasons upscale_FZJ-IDL-DA rcp
/gpfs/data/fs71966/amedvedova/publication_data/raw_data/ALP-12/rcp85/FZJ-IDL-DA/ALP-12_rcp85_FZJ-IDL-DA_10yr_1hr_complete.nc
allseasons upscale_KNMI rcp
/gpfs/data/fs71966/amedvedova/publication_data/raw_data/ALP-12/rcp85/KNMI/ALP-12_rcp85_KNMI_10yr_1hr_complete.nc
allseasons upscale_CLMcom-ETH-GAR rcp
/gpfs/data/fs71966/amedvedova/publication_data/raw_data/ALP-12/PGW-MPI-rcp85/CLMcom-ETH-GAR/ALP-12_pgw_CLMcom-ETH-GAR_10yr_1hr_complete.nc
allseasons upscale_IPSL hist
/gpfs/data/fs7

In [24]:
model_period_pairs = [
    (files_upscale_hist, "hist"),
    (files_upscale_rcp, "rcp"),
    ]

for season_months, season in zip(months, seasons):
            for files, period in model_period_pairs:
                for file_in in files:
                    make_all_datasets_from_hourly_data_parallel(file_in, season_months, season, metadata, folder_processed_data=folder_processed_data, savefiles=savefiles, period=period)

allseasons upscale_IPSL hist
/gpfs/data/fs71966/amedvedova/publication_data/raw_data/ALP-12/historical/IPSL/ALP-12_historical_IPSL_10yr_1hr_complete.nc
/gpfs/data/fs71966/amedvedova/publication_data/raw_data/ALP-12/historical/IPSL/ALP-12_historical_IPSL_10yr_1hr_complete.nc


ValueError: applied function returned data with an unexpected number of dimensions. Received 2 dimension(s) but expected 3 dimensions with names ('station_name', 'mean_daily_tas_bins', 'quantile'), from:

array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]], shape=(50, 277))

In [40]:
# kmscale = glob.glob(f'{folder_processed_data}/extended_*_measures_kmscale_*')
# upscale = glob.glob(f'{folder_processed_data}/extended_*_measures_upscale_*')

kmscale = glob.glob(f'{folder_processed_data}/various*kmscale_*')
upscale = glob.glob(f'{folder_processed_data}/various*upscale_*')

kmscale = [f for f in kmscale if "ensemble" not in f]
upscale = [f.replace('upscale', 'kmscale') for f in upscale]

set_a = {os.path.basename(f) for f in kmscale}
set_b = {os.path.basename(f) for f in upscale}

set_a - set_b


{'various_daily_stats_kmscale_CLMcom-ETH-GAR_rcp_dmean_allseasons.nc',
 'various_daily_stats_kmscale_CLMcom-KIT_rcp_dmean_allseasons.nc',
 'various_daily_stats_kmscale_CLMcom-WEGC_rcp_dmean_allseasons.nc',
 'various_daily_stats_kmscale_CNRM_rcp_dmean_allseasons.nc',
 'various_daily_stats_kmscale_FZJ-IDL-CA_rcp_dmean_allseasons.nc',
 'various_daily_stats_kmscale_FZJ-IDL-DA_rcp_dmean_allseasons.nc',
 'various_daily_stats_kmscale_HCLIMcom_rcp_dmean_allseasons.nc',
 'various_daily_stats_kmscale_IPSL_hist_dmean_allseasons.nc',
 'various_daily_stats_kmscale_IPSL_rcp_dmean_allseasons.nc',
 'various_daily_stats_kmscale_KNMI_rcp_dmean_allseasons.nc'}

In [36]:
ds = xr.open_dataset('/gpfs/data/fs71966/amedvedova/publication_data/raw_data/ALP-12/evaluation/ICTP/ALP-12_evaluation_ICTP_10yr_1hr_complete.nc')
ds

sh: 1: getfattr: not found


<xarray.Dataset> Size: 392MB
Dimensions:       (station_name: 277, time: 87672)
Coordinates:
  * station_name  (station_name) <U32 35kB 'Abtenau' ... 'Zwerndorf'
  * time          (time) datetime64[ns] 701kB 2000-01-01T00:30:00 ... 2009-12...
Data variables:
    pr            (station_name, time) float64 194MB ...
    tas           (station_name, time) float64 194MB ...
    elevation     (time) float64 701kB ...
    lat           (time) float64 701kB ...
    lon           (time) float64 701kB ...
    station_id    (time) float64 701kB ...
Attributes:
    units:    kg m-2 s-1

## Create ensemble files

In [ ]:
# create ensemble files: quantiles etc
# for resolution in ["kmscale", "upscale", "driving"]:
for resolution in ["upscale", ]:
    for season in seasons:
        print(resolution, season)
        for suffix in ["hdmean", "ddmean",]:
            for period in ["eval", "hist", "rcp"]:
                
                print(suffix, period)
                # get list of files
                quantiles_files = sorted(glob.glob(f"{folder_processed_data}/quantiles_{resolution}*_{period}_{suffix}_{season}.nc"))
                # get names of models from the filenames
                models = [f.split("/")[-1].split(f"_{suffix}_{season}.nc")[0].split(f"{resolution}_")[-1] for f in quantiles_files]
                # open as one dataset
                ds_combined = xr.open_mfdataset(quantiles_files, concat_dim="model", combine="nested", coords="minimal", data_vars="different")
                # assign name to dimension
                ds_combined["model"] = models
                # save file
                ds_combined.to_netcdf(f"{folder_processed_data}/quantiles_{resolution}_ensemble_{period}_{suffix}_{season}.nc")

        # create ensemble files: various daily stats
        for suffix in ["dmean", ]:
            for period in ["eval", "hist", "rcp"]:
                print(suffix, period)
                # get list of files
                various_daily_stats_files = sorted(glob.glob(f"{folder_processed_data}/various_daily_stats*{resolution}*{period}_{suffix}_{season}.nc"))
                # get names of models from the filenames
                models = [f.split("/")[-1].split(f"_{suffix}_{season}.nc")[0].split(f"{resolution}_")[-1] for f in various_daily_stats_files]
                # open as one dataset
                ds_combined = xr.open_mfdataset(various_daily_stats_files, concat_dim="model", combine="nested", coords="minimal", data_vars="different")
                # assign name to dimension
                ds_combined["model"] = models
                # save file
                ds_combined.to_netcdf(f"{folder_processed_data}/various_daily_stats_{resolution}_ensemble_{period}_{suffix}_{season}.nc")
        print()